# WatchTower — Full Attack Scenario Demo

**Real agent** with dangerous tools: `read_workspace_file`, `post_to_webhook`

| Service | Command |
|---------|---------|
| Backend | `uvicorn app.main:app --port 8000` |
| Dashboard | `npm run dev` → http://localhost:3000/dashboard |

```bash
# LLM — pick one
export OPENAI_API_KEY=sk-...
# OR (follows attacks more readily):
export LLM_PROVIDER=ollama && export OLLAMA_MODEL=llama3.2

# Your webhook.site URL for live exfil demo
export DEMO_WEBHOOK_URL=https://webhook.site/your-uuid-here
```

In [ ]:
# Set credentials via environment or examples/travel-agent/.env (never commit real keys)
# export OPENAI_API_KEY=sk-...
# export LLM_PROVIDER=ollama && export OLLAMA_MODEL=llama3.2
# export DEMO_WEBHOOK_URL=https://webhook.site/your-uuid-here


In [ ]:
import os, sys
from dotenv import load_dotenv
load_dotenv()

NOTEBOOK_DIR = os.getcwd()
ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "../.."))
sys.path[:0] = [os.path.join(ROOT, "backend"), os.path.join(ROOT, "watchtower"), NOTEBOOK_DIR]

from travel_agent import (
    build_travel_graph, invoke_unprotected, print_breach_report,
    show_detection, SCENARIOS, personalize_scenarios, run_scenario,
    require_llm, get_llm_provider,
)
from watchtower import SentinelClient

WEBHOOK = os.getenv("DEMO_WEBHOOK_URL", "https://webhook.site/your-uuid-here")
scenarios = personalize_scenarios(WEBHOOK)
graph = build_travel_graph(unprotected=True)
sentinel = SentinelClient("http://localhost:8000")

print(f"LLM: {require_llm()} | Webhook: {WEBHOOK}")
print(f"Scenarios loaded: {len(scenarios)}")

## Detection engine preview
WatchTower multi-signal analysis **before** the agent runs.

In [ ]:
print("── BENIGN ──")
show_detection(scenarios["safe_booking"]["prompt"])
print("\n── ATTACK: indirect injection ──")
show_detection(scenarios["indirect_injection"]["prompt"])
print("\n── ATTACK: SSRF ──")
show_detection(scenarios["ssrf_metadata"]["prompt"])
print("\n── ATTACK: PII harvest ──")
show_detection(scenarios["pii_harvesting"]["prompt"])

## Part A — WITHOUT WatchTower
Each cell runs the **real agent**. Secrets get read. Data POSTs to your webhook. Dashboard **unchanged**.

In [ ]:
# A1 · Normal booking — should succeed, no breach
r = invoke_unprotected(graph, scenarios["safe_booking"]["prompt"])
print(r["response"][:400])
print("Breach:", r.get("breach", False))

In [ ]:
# A2 · Prompt injection + exfiltration
breach = invoke_unprotected(graph, scenarios["prompt_injection_exfil"]["prompt"])
print_breach_report(breach)

In [ ]:
# A3 · Indirect injection (hidden inside a booking request)
breach = invoke_unprotected(graph, scenarios["indirect_injection"]["prompt"])
print_breach_report(breach)

In [ ]:
# A4 · Direct tool abuse — no "ignore instructions" needed
breach = invoke_unprotected(graph, scenarios["tool_abuse_direct"]["prompt"])
print_breach_report(breach)

In [ ]:
# A5 · PII harvesting (compliance impersonation)
breach = invoke_unprotected(graph, scenarios["pii_harvesting"]["prompt"])
print_breach_report(breach)

In [ ]:
# A6 · SSRF — cloud metadata endpoint
breach = invoke_unprotected(graph, scenarios["ssrf_metadata"]["prompt"])
print_breach_report(breach)

In [ ]:
# A7 · System prompt extraction
breach = invoke_unprotected(graph, scenarios["prompt_leak"]["prompt"])
print_breach_report(breach)

In [ ]:
# A8 · Social engineering (fake IT admin)
breach = invoke_unprotected(graph, scenarios["social_engineering"]["prompt"])
print_breach_report(breach)

In [ ]:
# A9 · Multi-vector (flight booking + silent exfil)
breach = invoke_unprotected(graph, scenarios["multi_vector"]["prompt"])
print_breach_report(breach)

## Part B — WITH WatchTower
Open dashboard tab. Same attacks — **blocked before tools run**. Graph goes red. Agent quarantined.

In [ ]:
await sentinel.reset()

ATTACK_KEYS = [
    "prompt_injection_exfil",
    "indirect_injection",
    "tool_abuse_direct",
    "pii_harvesting",
    "ssrf_metadata",
    "social_engineering",
    "multi_vector",
]

for key in ATTACK_KEYS:
    await sentinel.reset()
    out = await run_scenario(graph, key, protected=True, sentinel=sentinel, webhook_url=WEBHOOK)
    expected = scenarios[key]["expected"]
    blocked = out["blocked"]
    print(f"  {'✓' if blocked else '✗'} {key}: blocked={blocked} (expected threat: {expected})\n")

In [ ]:
# Safe request still works with WatchTower
await sentinel.reset()
safe = await run_scenario(graph, "safe_booking", protected=True, sentinel=sentinel)
print(safe["result"].response[:400] if safe["result"].response else safe)

In [ ]:
await sentinel.reset()
print("Dashboard reset — ready for live demo")

## Scenario reference

| Key | Attack type |
|-----|-------------|
| `safe_booking` | Benign |
| `prompt_injection_exfil` | Jailbreak + secrets exfil |
| `indirect_injection` | Hidden instruction in booking flow |
| `tool_abuse_direct` | Explicit tool chain, no jailbreak words |
| `pii_harvesting` | Bulk PII + pastebin |
| `ssrf_metadata` | AWS metadata IP probe |
| `prompt_leak` | System prompt theft |
| `social_engineering` | Fake IT admin authority |
| `multi_vector` | Booking cover + silent key exfil |